In [1]:
import os
import json
import fitz  # PyMuPDF
import pdfplumber
from docling.document_converter import DocumentConverter
from paddleocr import PaddleOCR
from PIL import Image
import io
import ollama
from pydantic import BaseModel, Field
from typing import List, Optional
from IPython.display import Markdown, display, JSON

# =====================================================================
# 1. DEFINISI SCHEMA PYDANTIC (COMPANION SCHEMA UNTUK NOCODB)
# =====================================================================

class SPHItem(BaseModel):
    item_name: str = Field(description="Nama item barang atau jasa")
    specs: Optional[str] = Field(None, description="Spesifikasi teknis item")
    qty: float = Field(description="Kuantitas/Volume")
    unit: str = Field(description="Satuan (misal: Unit, Bulan, Pcs, Lot)")
    unit_price: float = Field(description="Harga satuan (Rupiah)")
    total_price: float = Field(description="Total harga untuk item ini")

class SPHStructuredData(BaseModel):
    vendor_name: str = Field(description="Nama perusahaan / vendor penawar")
    sph_number: str = Field(description="Nomor surat penawaran harga")
    sph_date: str = Field(description="Tanggal surat penawaran")
    validity_period: Optional[str] = Field(None, description="Masa berlaku penawaran harga")
    payment_terms: Optional[str] = Field(None, description="Ketentuan atau termin pembayaran")
    items: List[SPHItem] = Field(description="Daftar rincian item / BoQ penawaran")
    subtotal: float = Field(description="Total sebelum PPN")
    ppn_amount: Optional[float] = Field(0.0, description="Nominal PPN jika ada")
    grand_total: float = Field(description="Total akhir penawaran termasuk pajak")

# =====================================================================
# 2. LAYER 1 & 2: ROUTER & DOCUMENT PARSERS
# =====================================================================

def is_scanned_pdf(pdf_path: str, char_threshold: int = 50) -> bool:
    """Mengecek apakah PDF berupa scan gambar atau teks digital."""
    doc = fitz.open(pdf_path)
    total_chars = sum(len(page.get_text().strip()) for page in doc)
    doc.close()
    return total_chars < char_threshold

def parse_with_docling(pdf_path: str) -> str:
    """Parser Lapis 1: Docling untuk PDF Digital Native."""
    print("🔹 Menggunakan [Docling] untuk parsing layout & tabel...")
    converter = DocumentConverter()
    result = converter.convert(pdf_path)
    return result.document.export_to_markdown()

# Inisialisasi PaddleOCR sekali (Bahasa Indonesia / English)
paddle_ocr_reader = None

def parse_with_paddleocr(pdf_path: str, max_pages: int = 3) -> str:
    """Parser Lapis 2: PaddleOCR untuk PDF Scan / Tanda Tangan."""
    global paddle_ocr_reader
    if paddle_ocr_reader is None:
        paddle_ocr_reader = PaddleOCR(use_angle_cls=True, lang="en")
        
    print(f"🔸 Menggunakan [PaddleOCR] untuk dokumen scan ({max_pages} hal)...")
    doc = fitz.open(pdf_path)
    markdown_lines = []
    
    pages_to_process = min(len(doc), max_pages)
    for i in range(pages_to_process):
        page = doc[i]
        pix = page.get_pixmap(dpi=150)
        img_bytes = pix.tobytes("png")
        
        # Jalankan PaddleOCR pada gambar halaman
        result = paddle_ocr_reader.ocr(img_bytes, cls=True)
        page_text = []
        if result and result[0]:
            for line in result[0]:
                text = line[1][0]
                page_text.append(text)
                
        markdown_lines.append(f"### Halaman {i+1}\n" + "\n".join(page_text))
        
    doc.close()
    return "\n\n---\n\n".join(markdown_lines)

# =====================================================================
# 3. LAYER 3: STRUCTURAL EXTRACTION DENGAN OLLAMA (QWEN 2.5 7B)
# =====================================================================

def extract_with_ollama(parsed_text: str, model_name: str = "qwen2.5:7b") -> SPHStructuredData:
    """
    Mengekstrak teks dokumen menjadi JSON terstruktur menggunakan Ollama & Pydantic.
    """
    print(f"🤖 Menjalankan Ekstraksi AI dengan Ollama [{model_name}]...")
    
    system_prompt = (
        "Anda adalah AI Data Extraction specialist untuk dokumen pengadaan dan SPH (Surat Penawaran Harga).\n"
        "Ekstrak semua informasi penting, tabel rincian harga/BoQ, dan metadata surat ke dalam format JSON yang tepat.\n"
        "Jangan menambahkan teks di luar JSON."
    )
    
    user_prompt = f"Berikut isi dokumen hasil parsing:\n\n{parsed_text}\n\nEkstrak ke format JSON sesuai schema."

    response = ollama.chat(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        format=SPHStructuredData.model_json_schema(),
        options={"temperature": 0.1}
    )
    
    json_str = response["message"]["content"]
    # Validasi kembali dengan model Pydantic
    return SPHStructuredData.model_validate_json(json_str)

# =====================================================================
# 4. ORKESTRATOR PIPELINE
# =====================================================================

def run_pipeline(pdf_path: str):
    print("=" * 70)
    print(f"📂 MEMPROSES DOKUMEN: {pdf_path}")
    print("=" * 70)
    
    if not os.path.exists(pdf_path):
        print(f"⚠️ File '{pdf_path}' tidak ditemukan!")
        return
        
    # Step 1: Deteksi tipe PDF
    if is_scanned_pdf(pdf_path):
        print("🔍 Tipe Dokumen: [SCANNED / IMAGE PDF]")
        parsed_md = parse_with_paddleocr(pdf_path, max_pages=3)
    else:
        print("🔍 Tipe Dokumen: [DIGITAL NATIVE PDF]")
        parsed_md = parse_with_docling(pdf_path)
        
    print("\n--- Ringkasan Hasil Parsing Dokumen (150 karakter) ---")
    print(parsed_md[:150] + "...\n")
    
    # Step 2: Ekstraksi dengan Ollama Qwen 2.5
    structured_data = extract_with_ollama(parsed_md)
    
    print("\n" + "=" * 70)
    print("✨ HASIL EKSTRAKSI TERSTRUKTUR (PYDANTIC -> NOCODB READY) ✨")
    print("=" * 70)
    
    # Cetak JSON yang sudah divalidasi
    print(json.dumps(structured_data.model_dump(), indent=2, ensure_ascii=False))
    
    return structured_data


/Users/sutriadik24/Downloads/buildAParser/.venv311/lib/python3.11/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


In [2]:
# 1. Test Dokumen SPH Digital (Docling + Ollama)
hasil_sph = run_pipeline("SPH Bapenda Jabar 2026.pdf")

# 2. Test Dokumen Scan (PaddleOCR + Ollama)
# hasil_scan = run_pipeline("NP_diskominfo_aceh_tamiang_2026 (1).pdf")


[2026-09-07 09:00:29,941] [    INFO] document.py:846 - detected formats: [<InputFormat.PDF: 'pdf'>]


📂 MEMPROSES DOKUMEN: SPH Bapenda Jabar 2026.pdf
🔍 Tipe Dokumen: [DIGITAL NATIVE PDF]
🔹 Menggunakan [Docling] untuk parsing layout & tabel...


[2026-09-07 09:00:30,246] [    INFO] document_converter.py:685 - Going to convert document batch...
[2026-09-07 09:00:30,246] [    INFO] document_converter.py:730 - Initializing pipeline for StandardPdfPipeline with options hash 23057d511012b65552d2c5ebefc52232
[2026-09-07 09:00:30,252] [    INFO] base_factory.py:115 - Loading plugin 'docling_defaults'
[2026-09-07 09:00:30,253] [    INFO] __init__.py:31 - Registered picture descriptions: ['picture_description_vlm_engine', 'vlm', 'api']
[2026-09-07 09:00:30,257] [    INFO] base_factory.py:115 - Loading plugin 'docling_defaults'
[2026-09-07 09:00:30,267] [    INFO] __init__.py:21 - Registered ocr engines: ['auto', 'easyocr', 'kserve_v2_ocr', 'nemotron-ocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
[2026-09-07 09:00:30,267] [    INFO] auto_ocr_model.py:62 - ocrmac cannot be used because ocrmac is not installed.
[2026-09-07 09:00:30,424] [    INFO] accelerator_utils.py:136 - Accelerator device: 'mps'
[INFO] 2026-09-07 09:00:30,436 [

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

[2026-09-07 09:00:34,636] [    INFO] transformers_engine.py:198 - Transformers engine ready (device=mps, dtype=torch.float32)
[2026-09-07 09:00:34,643] [    INFO] base_factory.py:115 - Loading plugin 'docling_defaults'
[2026-09-07 09:00:34,645] [    INFO] __init__.py:49 - Registered table structure engines: ['docling_tableformer', 'docling_tableformer_v2', 'granite_vision_table']
[2026-09-07 09:00:35,186] [    INFO] _client.py:1025 - HTTP Request: GET https://huggingface.co/api/models/docling-project/docling-models/revision/v2.3.0 "HTTP/1.1 200 OK"
[2026-09-07 09:00:35,221] [    INFO] accelerator_utils.py:136 - Accelerator device: 'mps'
[2026-09-07 09:00:35,825] [    INFO] base_pipeline.py:71 - Processing document SPH Bapenda Jabar 2026.pdf
[2026-09-07 09:00:43,227] [    INFO] document_converter.py:709 - Finished converting document SPH Bapenda Jabar 2026.pdf in 13.29 sec.



--- Ringkasan Hasil Parsing Dokumen (150 karakter) ---
Nomor:1241/00/BIS-05/BUT/2025

Bandung, 31 Desember 2025

Kepada Yth. MGR BIDDING MGT &amp; BUSINESS SUPPORT 2 REG II Graha Merah Putih Lantai 5 JI. J...

🤖 Menjalankan Ekstraksi AI dengan Ollama [qwen2.5:7b]...


[2026-09-07 09:01:17,231] [    INFO] _client.py:1025 - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"



✨ HASIL EKSTRAKSI TERSTRUKTUR (PYDANTIC -> NOCODB READY) ✨
{
  "vendor_name": "PT Bhakti Unggul Teknovasi",
  "sph_number": "1241/00/BIS-05/BUT/2025",
  "sph_date": "31 Desember 2025",
  "validity_period": null,
  "payment_terms": null,
  "items": [
    {
      "item_name": "Penyediaan Router Manage Service di 34 UPTD P3DW",
      "specs": null,
      "qty": 34.0,
      "unit": "Unit/Bulan",
      "unit_price": 999500.0,
      "total_price": 407796000.0
    },
    {
      "item_name": "Jasa Manage Service Router Cisco 1941",
      "specs": null,
      "qty": 34.0,
      "unit": "Unit/Bulan",
      "unit_price": 141500.0,
      "total_price": 57732000.0
    },
    {
      "item_name": "Dokumentasi & Pelaporan Renewal UTM Sophos XGS 2300 1Year",
      "specs": null,
      "qty": 34.0,
      "unit": "Unit/Bulan",
      "unit_price": 395000.0,
      "total_price": 161160000.0
    }
  ],
  "subtotal": 626688000.0,
  "ppn_amount": 0.0,
  "grand_total": 1326738000.0
}
